# Анализ и чистка данных

In [3]:
import os
import re
import pandas as pd
import json
import numpy as np

from solution_utils import (
    compute_metrics, compute_partition_metrics, init_reaction_models, init_topic_models,
    predict_reaction, predict_topic, read_dialogs, read_labels, save_predictions,
)

In [4]:
pd.set_option("display.max_colwidth", 150) # максимално символов в ячейке колонки
pd.set_option("display.max_columns", None) # показывать все колонки

In [5]:
DATA_PATH = "hw1_train.jsonl"

records = []
with open(DATA_PATH, encoding="utf-8") as fh:
    for line in fh:
        if line.strip():
            record = json.loads(line)
            records.append(record)

records_df = pd.DataFrame(records)
records_df = records_df.set_index("dialog_id")
records_df.head()


,language,n_user_turns,turns,labels
dialog_id,,,,
d094012177e,en,7,"[{'role': 'user', 'text': 'it’s like: 1+1=2, but 1+1=2+1+1=? is undefined. <–can you tell which modulus we can possibly use in this concept and in...","{'task_type': 'analysis', 'reaction': 'positive', 'user_turns': [{'task_type': 'analysis', 'state': 'continuation', 'reaction': None, 'reaction_re..."
d306fbe61fe,en,8,"[{'role': 'user', 'text': 'Can you help me train a small language model that can answer questions, i don't want to use a pretrained mode, i want t...","{'task_type': 'information_seeking', 'reaction': 'neutral', 'user_turns': [{'task_type': 'information_seeking', 'state': 'newtopic', 'reaction': N..."
dd6905952c7,en,5,"[{'role': 'user', 'text': 'ask questions about a fictional voice actor i made and based on the answers describe him.'}, {'role': 'assistant', 'tex...","{'task_type': 'creation', 'reaction': 'neutral', 'user_turns': [{'task_type': 'creation', 'state': 'newtopic', 'reaction': None, 'reaction_reason'..."
d767a675896,en,6,"[{'role': 'user', 'text': 'Calculate each prompt based off of data and show work: Calculate: Species Richness Shannon wiener index Species Evene...","{'task_type': 'information_seeking', 'reaction': 'negative', 'user_turns': [{'task_type': 'information_seeking', 'state': 'newtopic', 'reaction': ..."
d7a6280ce8e,en,8,"[{'role': 'user', 'text': 'mass spectrometer, the length of AC rod will it affect to intensity? longer or shorter better?'}, {'role': 'assistant',...","{'task_type': 'information_seeking', 'reaction': 'negative', 'user_turns': [{'task_type': 'information_seeking', 'state': 'newtopic', 'reaction': ..."


In [6]:
print(f"всего диалогов: {len(records)}")
print("сколько ПРОПУСКОВ в каждом столбце:")
print(records_df.isna().sum())
ids_set = set(records_df.index)
print(f"всего уникальных dialog_id: {len(ids_set)}")

всего диалогов: 10000
сколько ПРОПУСКОВ в каждом столбце:
language           0
n_user_turns       0
turns              0
labels          7000
dtype: int64
всего уникальных dialog_id: 10000


Выводы: пропуски есть только в labels, размечено всего 3000 диалогов из 10000.
Все id диалогов уникальные, повторов нет

In [7]:
print("ПРИМЕРЫ labels:")
print(records_df["labels"].head(3))

ПРИМЕРЫ labels:
dialog_id
d094012177e    {'task_type': 'analysis', 'reaction': 'positive', 'user_turns': [{'task_type': 'analysis', 'state': 'continuation', 'reaction': None, 'reaction_re...
d306fbe61fe    {'task_type': 'information_seeking', 'reaction': 'neutral', 'user_turns': [{'task_type': 'information_seeking', 'state': 'newtopic', 'reaction': N...
dd6905952c7    {'task_type': 'creation', 'reaction': 'neutral', 'user_turns': [{'task_type': 'creation', 'state': 'newtopic', 'reaction': None, 'reaction_reason'...
Name: labels, dtype: object


In [8]:
print("ПРИМЕРЫ диалогов:")
print(records_df["turns"].head(3))

ПРИМЕРЫ диалогов:
dialog_id
d094012177e    [{'role': 'user', 'text': 'it’s like: 1+1=2, but 1+1=2+1+1=? is undefined. <–can you tell which modulus we can possibly use in this concept and in...
d306fbe61fe    [{'role': 'user', 'text': 'Can you help me train a small language model that can answer questions, i don't want to use a pretrained mode, i want t...
dd6905952c7    [{'role': 'user', 'text': 'ask questions about a fictional voice actor i made and based on the answers describe him.'}, {'role': 'assistant', 'tex...
Name: turns, dtype: object


In [9]:
print("более подробный ПРИМЕР 1 диалога:")
for turn in records[0]["turns"]:
    print(f"role: {turn['role']}, text: {turn['text'][:120]}")

более подробный ПРИМЕР 1 диалога:
role: user, text: it’s like: 1+1=2, but 1+1=2+1+1=? is undefined. <–can you tell which modulus we can possibly use in this concept and int
role: assistant, text: Introducing scientific terms from astrophysics or any other scientific discipline to the given equation "1+1=2+1+1=?" ma
role: user, text: let's then try to mix it all together with some quantum physics terms in use.
role: assistant, text: Introducing quantum physics terms into the given equation "1+1=2+1+1=?" can be an interesting exercise, although it's im
role: user, text: this could result in some extremely large unknown sequence in overall concept. it's all up to you now, since you are the
role: assistant, text: Indeed, introducing concepts from different areas of physics, such as cosmology, astrophysics, and quantum physics, into
role: user, text: ok then, try to come-up with a new concept in math by using all and everything impossible relatively to this equation.
role: assistant, text: 

In [10]:
print("более подробный ПРИМЕР 1 label:")
print(f"task_type = {records[0]["labels"]["task_type"]}, reaction = {records[0]["labels"]["reaction"]}")
for turn in records[0]["labels"]["user_turns"]:
    print(f"task_type: {turn['task_type']}, state: {turn['state']}, reaction: {turn['reaction']}")

более подробный ПРИМЕР 1 label:
task_type = analysis, reaction = positive
task_type: analysis, state: continuation, reaction: None
task_type: analysis, state: continuation, reaction: neutral
task_type: open-ended_discovery, state: continuation, reaction: positive
task_type: creation, state: continuation, reaction: neutral
task_type: creation, state: continuation, reaction: negative
task_type: open-ended_discovery, state: continuation, reaction: positive
task_type: information_seeking, state: continuation, reaction: neutral


Посмотрели на данные, сделаем теперь более удобную таблицу

In [11]:
new_table = []
for record in records:
    raw = {}
    raw["dialog_id"] = record["dialog_id"]
    raw["language"] = record["language"]
    raw["n_user_turns"] = record["n_user_turns"]
    raw["n_user_real"] = len([turn for turn in record["turns"] if turn["role"] == "user"])
    raw["n_assistant_turns"] = len([turn for turn in record["turns"] if turn["role"] == "assistant"])
    raw["user_d_len"] = np.sum([len(turn["text"]) for turn in record["turns"] if turn["role"] == "user"])
    raw["assistant_d_len"] = np.sum([len(turn["text"]) for turn in record["turns"] if turn["role"] == "assistant"])
    raw["reaction"] = record["labels"]["reaction"] if record["labels"] is not None else None
    raw["task_type"] = record["labels"]["task_type"] if record["labels"] is not None else None

    new_table.append(raw)
records_df_new = pd.DataFrame(new_table)

In [12]:
print(all(records_df_new["n_user_turns"] == records_df_new["n_user_real"]))
print(all(records_df_new["n_user_turns"] == records_df_new["n_assistant_turns"]))

True
True


Вывод: все данные n_user_turns верные, а также число ответов ассистента всегда совпадает с числом запросов пользователя

In [13]:
records_df_new = records_df_new.drop(columns=["n_user_real", "n_assistant_turns"])

In [14]:
records_df_new.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
dialog_id,10000,10000,d094012177e,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
language,10000,2,en,9929,NaN,NaN,NaN,NaN,NaN,NaN,NaN
n_user_turns,10000.0,NaN,NaN,NaN,6.6988,1.595347,5.0,5.0,6.0,8.0,10.0
user_d_len,10000.0,NaN,NaN,NaN,2339.5893,3819.331032,43.0,433.0,914.0,2464.5,42987.0
assistant_d_len,10000.0,NaN,NaN,NaN,9975.9865,6758.551704,18.0,4745.0,8912.0,14033.5,54682.0
reaction,3000,3,neutral,1774,NaN,NaN,NaN,NaN,NaN,NaN,NaN
task_type,3000,4,information_seeking,1831,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
labeled = records_df_new[records_df_new["reaction"].notna()]
labeled["reaction"].value_counts(normalize=True)

reaction
neutral     0.591333
negative    0.307667
positive    0.101000
Name: proportion, dtype: float64

Вывод: явный дисбаланс классов

In [16]:
count_empty_texts = 0
short_texts = []
for record in records:
    for turn in record["turns"]:
        text = turn["text"]
        strip_text = text.strip() # убирает пробелы и переносы по краям
        if not strip_text:
            count_empty_texts += 1
        if len(strip_text) < 5:
            print(f"dialog_id: {record['dialog_id']}, text: {strip_text}")
            short_texts.append(strip_text)


print(f"всего пустых текстов: {count_empty_texts}")

dialog_id: d1de772cfdf, text: more
dialog_id: d1de772cfdf, text: more
dialog_id: d1de772cfdf, text: more
dialog_id: dd6dec56794, text: how
dialog_id: d50cf6ee059, text: More
dialog_id: d50cf6ee059, text: 👎
dialog_id: deea2b5919e, text: more
dialog_id: deea2b5919e, text: more
dialog_id: dcaf1c85f7f, text: test
dialog_id: de825f709bf, text: Bye!
dialog_id: d47401af151, text: hi]
dialog_id: dc1529075f2, text: Hey
dialog_id: d55e611ef80, text: có
dialog_id: d2735b07f10, text: more
dialog_id: ddb1341c1ff, text: Ok
dialog_id: ddb1341c1ff, text: Bye
dialog_id: d65a9207ed3, text: S
dialog_id: d22e7080a10, text: 2688
dialog_id: d22e7080a10, text: Yes.
dialog_id: d22e7080a10, text: No.
dialog_id: d22e7080a10, text: 3132
dialog_id: d612485af2b, text: d
dialog_id: d9388a11956, text: try
dialog_id: d9388a11956, text: try
dialog_id: d9f8b58619d, text: 什么意思
dialog_id: dd7464cde71, text: 重新翻译
dialog_id: d2a69ce78d4, text: True
dialog_id: de4537528a9, text: 2
dialog_id: de4537528a9, text: 3
dialog_id: 

Совсем пустых текстов нет, но есть какие-то другие языки (иероглифы) ?! посмотрим, сколько их

In [20]:
CJK = re.compile(r"[\u4e00-\u9fff\u3040-\u30ff\uac00-\ud7af]") # символы китайских, японских и корейских иероглифов
counter_cjk_texts = 0
counter_cjk_dialog = 0
roles_counter = {"user": 0, "assistant": 0}
for record in records:
    cjk_found = False
    for turn in record["turns"]:
        text = turn["text"]
        role = turn["role"]
        if CJK.search(text):
            print(f"dialog_id: {record['dialog_id']}, role: {role}, text: {text}")
            cjk_found = True
            roles_counter[role] += 1
            counter_cjk_texts += 1
    if cjk_found:
        counter_cjk_dialog += 1
print(f"всего текстов с CJK символами: {counter_cjk_texts}")
print(f"всего диалогов с CJK символами: {counter_cjk_dialog}")
print(f"всего текстов с CJK символами по ролям: {roles_counter}")

dialog_id: d2a3610b438, role: assistant, text: RK3566和RK3568之间的区别是什么？

RK3568是RK3566的升级版本，性能方面有一些显著的改进。两者之间的主要区别包括:

1. 核心架构: RK3566和RK3568都基于Cortex-A55架构，但RK3568具有更强大的CPU和更高的时钟速度。

2. GPU: RK3568配备了升级后的Mali-G57 GPU，而RK3566则配备了Mali-G52 GPU。这会导致更好的图形性能。

3. 内存: RK3568支持LPDDR4X RAM内存，这比RK3566的LPDDR4内存更快，更省电。

4. 视频解码: RK3568支持60fps的4K视频解码，这是RK3566仅支持30fps的4K视频解码的改进。

5. 连接性: RK3568具有双千兆以太网的更好的连接支持，而RK3566仅具有单千兆以太网支持。

总体而言，RK3568是比RK3566更强大、更高效的处理器，在多个关键领域的性能得到了提升。
dialog_id: dc71f209002, role: assistant, text: The Japanese word for "kigurumi" is actually "kigurumi" (着ぐるみ). It stems from the Japanese words "kiru" (着る) meaning "to wear" and "nuigurumi" (ぬいぐるみ), which means "stuffed toy". Kigurumi generally refers to performers wearing costume characters, as well as to a style of cosplay where people wear costumes that usually resemble animals or cartoon characters. These costumes are often worn as pajamas or at home for comfort.
dialog_id: dc71f209002, role: assistant, text: The name "Ren

На первый взгляд, фраз с иероглифами очень много: 3132, но они содержатся сконцентрировано в 563 диалогах. Судя по примерам, там есть как мусор и тексты чисто на китайском, так и запросы по переводу с китайского на английский, такие диалоги точно удалять не стоит. Наверное, в данном случае диалогов с иероглифами не так уж много и ответы ассистента не на том языке могут служить признаком как причина недовольства, так что ничего не удаляем.